# Delta Lake com PySpark

Aqui a gente demonstra as operações principais do Delta Lake usando PySpark.
O dataset é de vendas de um e-commerce fictício.

Operações cobertas: criação da tabela, INSERT, UPDATE, DELETE e Time Travel.

## Criando a SparkSession com suporte ao Delta Lake

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, lit
from delta import *
from delta.tables import DeltaTable

# configurando a sessão com as extensões do delta
builder = SparkSession.builder \
    .appName("Delta Lake Demo") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark versao: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

## Definindo o schema e os dados iniciais

In [ ]:
DELTA_PATH = "./delta-warehouse/vendas"

schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("produto",    StringType(),  True),
    StructField("categoria",  StringType(),  True),
    StructField("quantidade", IntegerType(), True),
    StructField("preco",      DoubleType(),  True),
    StructField("data_venda", StringType(),  True),
    StructField("vendedor",   StringType(),  True),
    StructField("status",     StringType(),  True),
])

dados_iniciais = [
    (1,  "Notebook Dell",     "Eletronicos", 2, 3500.00, "2024-01-15", "Ana Silva",    "entregue"),
    (2,  "Camiseta Polo",     "Roupas",      5,   89.90, "2024-01-20", "Bruno Costa",  "entregue"),
    (3,  "Tenis Running",     "Calcados",    1,  450.00, "2024-02-01", "Carlos Lima",  "pago"),
    (4,  "Mouse Gamer",       "Eletronicos", 3,  250.00, "2024-02-10", "Diana Ramos",  "entregue"),
    (5,  "Calca Jeans",       "Roupas",      2,  199.90, "2024-02-14", "Eduardo Melo", "entregue"),
    (6,  "Smartphone X",      "Eletronicos", 1, 2800.00, "2024-02-20", "Ana Silva",    "pago"),
    (7,  "Mochila Viagem",    "Acessorios",  1,  350.00, "2024-03-01", "Bruno Costa",  "pendente"),
    (8,  "Teclado Mecanico",  "Eletronicos", 1,  450.00, "2024-03-05", "Carlos Lima",  "pago"),
    (9,  "Vestido Floral",    "Roupas",      3,  159.90, "2024-03-10", "Diana Ramos",  "entregue"),
    (10, "Monitor 27pol",     "Eletronicos", 1, 1800.00, "2024-03-15", "Eduardo Melo", "pendente"),
    (11, "Sandalia Couro",    "Calcados",    2,  280.00, "2024-03-18", "Ana Silva",    "entregue"),
    (12, "Camera DSLR",       "Eletronicos", 1, 4200.00, "2024-03-20", "Bruno Costa",  "pago"),
    (13, "Jaqueta Denim",     "Roupas",      1,  320.00, "2024-03-22", "Carlos Lima",  "cancelado"),
    (14, "Headset USB",       "Eletronicos", 2,  180.00, "2024-03-25", "Diana Ramos",  "cancelado"),
    (15, "Relogio Analogico", "Acessorios",  1,  520.00, "2024-03-28", "Eduardo Melo", "pendente"),
]

print(f"Total de registros: {len(dados_iniciais)}")

## INSERT — carga inicial dos dados

Cria o DataFrame e grava no formato Delta com `overwrite`.

In [ ]:
df_inicial = spark.createDataFrame(dados_iniciais, schema)

df_inicial.write.format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

df_leitura = spark.read.format("delta").load(DELTA_PATH)
print(f"Registros inseridos: {df_leitura.count()}")
df_leitura.show(truncate=False)

### INSERT — append de novos pedidos

Simulando chegada de novos pedidos com `mode('append')`.

In [ ]:
novos_pedidos = [
    (16, "Fone Bluetooth", "Eletronicos", 1, 299.90, "2024-04-01", "Ana Silva",   "pendente"),
    (17, "Bolsa de Couro", "Acessorios",  1, 480.00, "2024-04-02", "Bruno Costa", "pendente"),
    (18, "Tenis Social",   "Calcados",    1, 390.00, "2024-04-03", "Carlos Lima", "pendente"),
]

df_novos = spark.createDataFrame(novos_pedidos, schema)

df_novos.write.format("delta") \
    .mode("append") \
    .save(DELTA_PATH)

df_atual = spark.read.format("delta").load(DELTA_PATH)
print(f"Total de registros agora: {df_atual.count()}")
df_atual.filter(col("id") >= 16).show(truncate=False)

## UPDATE — atualizando registros

O Delta Lake faz UPDATE sem modificar os arquivos antigos — ele cria um novo snapshot.

In [ ]:
deltaTable = DeltaTable.forPath(spark, DELTA_PATH)

print("antes do update — pedidos pendentes:")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("status") == "pendente") \
    .select("id", "produto", "vendedor", "status") \
    .show(truncate=False)

# atualiza todos os pedidos pendentes para pago
deltaTable.update(
    condition = col("status") == "pendente",
    set       = {"status": lit("pago")}
)

print("depois do update:")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("id").isin([7, 10, 15, 16, 17, 18])) \
    .select("id", "produto", "vendedor", "status") \
    .show(truncate=False)

In [ ]:
# segundo update: pedidos pagos mais antigos passam para entregue
deltaTable.update(
    condition = (col("status") == "pago") & (col("data_venda") < "2024-03-01"),
    set       = {"status": lit("entregue")}
)

print("contagem por status apos os dois updates:")
spark.read.format("delta").load(DELTA_PATH) \
    .groupBy("status").count() \
    .orderBy("status") \
    .show()

## DELETE — removendo registros

Remove os pedidos cancelados. O Delta mantém os arquivos antigos para o Time Travel funcionar.

In [ ]:
print("registros que serao deletados:")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("status") == "cancelado") \
    .show(truncate=False)

deltaTable.delete(condition = col("status") == "cancelado")

df_pos_delete = spark.read.format("delta").load(DELTA_PATH)
print(f"Registros restantes: {df_pos_delete.count()}")
df_pos_delete.groupBy("status").count().orderBy("status").show()

## Time Travel — consultando versões anteriores

Um dos recursos mais interessantes do Delta Lake é poder consultar o estado da tabela em qualquer versão anterior.

In [ ]:
# historico de operacoes na tabela
print("historico de operacoes:")
deltaTable.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

In [ ]:
# lendo a versao 0 — estado original antes de qualquer modificacao
print("versao 0 da tabela (antes de qualquer update/delete):")
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(DELTA_PATH)

print(f"registros na versao 0: {df_v0.count()}")
df_v0.groupBy("status").count().orderBy("status").show()

# comparando os cancelados
cancelados_v0     = df_v0.filter(col("status") == "cancelado").count()
cancelados_atual  = spark.read.format("delta").load(DELTA_PATH).filter(col("status") == "cancelado").count()
print(f"cancelados na v0: {cancelados_v0}  |  cancelados na versao atual: {cancelados_atual}")

## Estado final e estatísticas

In [ ]:
from pyspark.sql.functions import sum as spark_sum, avg, count

df_final = spark.read.format("delta").load(DELTA_PATH)

print("tabela final:")
df_final.show(truncate=False)

print("estatisticas por categoria:")
df_final.groupBy("categoria") \
    .agg(
        count("id").alias("total_pedidos"),
        spark_sum("quantidade").alias("total_itens"),
        spark_sum((col("preco") * col("quantidade"))).alias("receita_total"),
        avg("preco").alias("preco_medio")
    ) \
    .orderBy("receita_total", ascending=False) \
    .show(truncate=False)

spark.stop()